<a href="https://colab.research.google.com/github/TomographicImaging/gVXR-Tutorials/blob/main/notebooks/segmentation-to-CT_scan-simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
#
#  Copyright 2026 United Kingdom Research and Innovation
#
#  Licensed under the Apache License, Version 2.0 (the "License");
#  you may not use this file except in compliance with the License.
#  You may obtain a copy of the License at
#
#      http://www.apache.org/licenses/LICENSE-2.0
#
#  Unless required by applicable law or agreed to in writing, software
#  distributed under the License is distributed on an "AS IS" BASIS,
#  WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
#  See the License for the specific language governing permissions and
#  limitations under the License.
#
#   Authored by:    Franck Vidal (UKRI-STFC)
#   Data by:    Matthew Jones (UCL)

![gVXR](https://github.com/TomographicImaging/gVXR-Tutorials/blob/main/img/Logo-transparent-small.png?raw=1)

# From image segmentation to CT simulation

This notebook shows how to:
1. create a digital twin using the digital twin framework introduced in gVXR 2.1.0;
2. load a segmented image and you it to create a multi-part sample; it relies on functions built in gVXR only, i.e. no third-party software is required; the functionality is multi-threaded to boost performances;
3. simulate a CT scan acquisition;
4. use CIL to recontruct the corresponding CT volume.

![Segmented data](../data/labels.png)

<div class="alert alert-block alert-warning">
    <b>Note:</b> Make sure the Python packages are already installed. See <a href="../README.md">README.md</a> in the root directory of the repository. If you are running this notebook from Google Colab, please run the cell below to install the package with `!pip install gvxr`
</div>

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !apt-get install libnvidia-gl-575
    !pip install -q condacolab
    import condacolab
    condacolab.install()

    !conda install -y -c conda-forge -c https://software.repos.intel.com/python/conda -c ccpi cil=24.3.0 ipp=2021.12 tigre

    !pip install gvxr

In [ ]:
import os # Create the output directory if necessary
import numpy as np # Who does not use Numpy?

from tifffile import  imread, imwrite
import matplotlib.pyplot as plt # Plotting
import matplotlib.cm as cm
plt.style.use('tableau-colorblind10')

from tqdm.notebook import tqdm

#  CT simulation
from gvxrPython3 import gvxr
from gvxrPython3.twins.utils import getDigitalTwinList, createDigitalTwin
from gvxrPython3.gVXRDataReader import *

# CT reconstruction
from cil.plugins.astra import FBP
from cil.io import TIFFWriter

from cil.processors import TransmissionAbsorptionConverter
from cil.utilities.display import show_geometry

from utils import pad_image, crop_image

## Getting the data ready

Where to save the data.

In [ ]:
output_path = "./output_data/segmentation-to-CT_scan-simulation"
if not os.path.exists(output_path):
    os.makedirs(output_path)

## 1. Set the simulation environment

In [ ]:
# Create an OpenGL context
print("Create an OpenGL context")
gvxr.createOpenGLContext(-1, 4, 6, 41)

# 2. Create a digital twin

We first show the list of twins that are included in the framework.

In [ ]:
available_twins = getDigitalTwinList()

# Process every twin
for i, twin in enumerate(available_twins):
    print("%i. %s" % (i+1, twin["name"]))
    print("\t- Available at:", twin["facility"])

    # Process every beam
    print("\t- Available beams:")
    for j, beam_label in enumerate(twin["beams"]):
        beam = twin["beams"][beam_label]
        beam_type = beam["beam_type"]
        print("\t\t%i. \"%s\"" % (j+1, beam_label))
        print("\t\t\t- Type:", beam_type)

        if beam_type == "monochromatic":
            print("\t\t\t- Energy range [in keV]:", beam["keV"])
        elif beam_type == "tube":
            print("\t\t\t- Voltage range [in kV]:", beam["kV"])

    detectors = twin["detector"]
    print("\t- Pixel pitch [in um]:", detectors["pixel_pitch"])

    resolutions = detectors["resolutions"]
    if len(resolutions) == 1:
        print("\t- Available resolutions: %ix%i" % (resolutions[0][0], resolutions[0][1]))
    else:
        print("\t- Available resolutions:")
        for k, resolution in enumerate(detectors["resolutions"]):
            print("\t\t%i. %ix%i" % (k+1, resolution[0], resolution[1]))
    if detectors["scintillator"] != None and detectors["scintillator"] != "":
        print("\t- Scintillator: %i um of %s" %
              (detectors["scintillator"]["thickness"], detectors["scintillator"]["material"])
        )
    else:
        print("\t- Scintillator: unknown")


We'll make use of the "XT H 225 ST 2x" twin.

In [ ]:
twin = createDigitalTwin(name="XT H 225")

Print the characteristics of this twin

In [ ]:
print(twin)

We can change the voltage within its range of validity

In [ ]:
beam_name = list(twin.specification.beams)[0]
print("Valid voltage range [in kV]:", twin.specification.beams[beam_name].kV)


We will use 225 kV.

In [ ]:
twin.beam.kV = 225

## Tube current

Check what current values are allowed and what the default value is

In [ ]:
twin_max_uA = max(twin.specification.beams[beam_name].uA)
twin_min_uA = min(twin.specification.beams[beam_name].uA)
default_uA_value = twin.beam.uA

print("%f <= %f <= %f" % (twin_min_uA, default_uA_value, twin_max_uA))

Change the current to 160 uA, the exposure to 0.5 seconds and enable photonic noise

In [ ]:
twin.beam.uA = 160
twin.detector.exposure = 0.5
gvxr.enablePoissonNoise()

Record the original resolution

In [ ]:
twin.apply()
original_resolution = gvxr.getDetectorNumberOfPixels()
original_pixel_spacing_in_mm = gvxr.getDetectorPixelSpacing("mm")

Apply the current setting

In [ ]:
# Downsample the detector to speed up the calculations
downsampling_factor = 1
if downsampling_factor > 1:

    for i, resolution in enumerate(twin.specification.detector.resolutions):
        twin.specification.detector.resolutions[i] = [
            round(resolution[0] / downsampling_factor),
            round(resolution[1] / downsampling_factor)
        ]

    twin.specification.detector.pixel_pitch *= downsampling_factor

In [ ]:
twin.apply()

Add 1 mm of Cu as filtration

In [ ]:
gvxr.addFilter("Cu", 1.0, "mm")

## 3. Create the sample from a segmented image

Read the image

In [ ]:
labels = imread("../../data/morning/seg1.tif")
original_pixel_size = [0.01307412, 0.01307412]
unit = "mm"

Display the image

In [ ]:
fig = plt.figure()
im = plt.imshow(labels,
                origin='upper',
                extent=(0.0, original_pixel_size[0] * labels.shape[1],
                        0.0, original_pixel_size[1] * labels.shape[0]),
)
plt.xlabel("Pixel position [" + unit + "]")
plt.ylabel("Pixel position [" + unit + "]")
fig.colorbar(im)
plt.show()

In [ ]:
labels_cropped = crop_image(labels)
fig = plt.figure()
im = plt.imshow(labels_cropped,
                origin='upper',
                extent=(0.0, original_pixel_size[0] * labels_cropped.shape[1],
                        0.0, original_pixel_size[1] * labels_cropped.shape[0]),
)
plt.xlabel("Pixel position [" + unit + "]")
plt.ylabel("Pixel position [" + unit + "]")
fig.colorbar(im)
plt.show()

imwrite(os.path.join(output_path, "labels-cropped.tif"), labels_cropped)

Make sure it is a 3D image

In [ ]:
if len(labels_cropped.shape) == 2:
    labels = np.repeat(labels_cropped[np.newaxis, :, :], 5, axis=0)
    # labels.shape = [1, *labels.shape]

In [ ]:
material_composition = {

    1: {
        'name': "Al",
        'material type': 'element',
        'material': 'Al',
        'density': 2.70
    },

    2: {
        'name': "Cu-lower-density",
        'material type': 'element',
        'material': 'Cu',
        'density': 1.87
    },

    3: {
        'name': "NMC811",
        'material type': 'mixture',
        'material': 'NMC811',
        'materials': ["Li", "Ni", "Mn", "Co"],
        'weights': [0.8 * 0.1057, 0.8 * 0.8943, 0.1, 0.1],
        'density': 2.61
    },

    4: {
        'name': "Cu-higher-density",
        'material type': 'element',
        'material': 'Cu',
        'density': 8.96
    },

    5: {
        'name': "Fe70Cr19Ni10Mn1",
        'material type': 'mixture',
        'material': 'Fe70Cr19Ni10Mn1',
        'density': 8.00
    }
}

Process the segmentation to create iso surfaces

In [ ]:
gvxr.removePolygonMeshesFromSceneGraph()

gvxr.emptyMesh("sample")

for label in tqdm(np.unique(labels)):

    if label in material_composition.keys():
        selected_material = material_composition[label]
        mesh_label = material_composition[label]['name']

        print("Process", mesh_label)

        # Select the structure
        binary_image = (labels == label).astype(np.uint8)

        stl_fname = os.path.join(output_path, mesh_label+".stl")

        # Load the existing STL file
        if os.path.exists(stl_fname):
            gvxr.loadMeshFile(mesh_label, stl_fname, unit, False, "sample")
        # Apply the Marching cubes
        else:
            # Let' make it a cube
            image_size = np.average([
                labels.shape[2] * original_pixel_size[0],
                labels.shape[1] * original_pixel_size[0]
            ])

            voxel_size = [
                original_pixel_size[0],
                original_pixel_size[1],
                image_size / labels.shape[0]
            ]

            # gVXR 2.1.0 or above
            if gvxr.getMajorVersionOfSimpleGVXR() >= 2 and gvxr.getMinorVersionOfSimpleGVXR() >= 1:
                gvxr.makeIsoSurface(mesh_label,
                    binary_image,
                    1,
                    0, 0, 0,
                    *voxel_size,
                    unit,
                    "sample"
                )
            else:
                gvxr.makeIsoSurface(mesh_label, # Mesh label
                    1, # Iso-value
                    binary_image.flatten(), # Dataset
                    binary_image.shape[2], binary_image.shape[1], binary_image.shape[0], # Volume size in voxel
                    *voxel_size, # Voxel spacing
                    unit, # Unit of length
                    "sample"
                )

            # Save the mesh as an STL file
            gvxr.saveSTLfile(mesh_label, stl_fname)

        # Set the material
        if selected_material['material type'].upper() == 'ELEMENT':
            print("\tUse element", material_composition[label]["material"])
            gvxr.setElement(mesh_label, material_composition[label]["material"])

            if "density" in selected_material:
                gvxr.setDensity(mesh_label, selected_material["density"], "g/cm3")

        elif selected_material['material type'].upper() == 'COMPOUND':
            print("\tUse compound", material_composition[label]["material"])
            gvxr.setCompound(mesh_label, material_composition[label]["material"])
            gvxr.setDensity(mesh_label, selected_material["density"], "g/cm3")
        
        elif selected_material['material type'].upper() == 'MIXTURE':
            print("\tUse mixture", material_composition[label]["material"])

            if "materials" in selected_material and "weights" in selected_material:
                gvxr.setMixture(mesh_label, selected_material["materials"], selected_material["weights"])
            else:
                gvxr.setMixture(mesh_label, material_composition[label]["material"])

            gvxr.setDensity(mesh_label, selected_material["density"], "g/cm3")
        
        else:
            raise IOError("Invalid material type")

        # Add the material
        gvxr.addPolygonMeshAsInnerSurface(mesh_label)
    else:
        print(label, "is not in", material_composition)


Move the sample to maximise the magnification

In [ ]:
optimal_sod = gvxr.getBestSOD("sample", "cm")
print("Optimal SOD:", optimal_sod)
gvxr.applySOD("sample", optimal_sod, "cm")

bbox_centre = gvxr.getNodeAndChildrenBoundingBoxCentre("root", unit)

In [ ]:
xray_image = np.array(gvxr.computeXRayImage()) / gvxr.getTotalEnergyWithDetectorResponse()
pixel_size_with_magnification = gvxr.getProjectionPixelSize("mm")
extent = (
    0, xray_image.shape[1] * pixel_size_with_magnification[0],
    0, xray_image.shape[0] * pixel_size_with_magnification[1]
)
plt.imshow(xray_image, extent=extent, cmap="gray")
plt.colorbar()
plt.xlabel("Pixel position [in mm]")
plt.ylabel("Pixel position [in mm]")
plt.show()

In [ ]:
lbuffer_set = {}

for label in tqdm(np.unique(labels)):

    if label in material_composition.keys():
        selected_material = material_composition[label]
        mesh_label = material_composition[label]['name']
        lbuffer_set[mesh_label] = np.array(gvxr.computeLBuffer(mesh_label), dtype=np.single)

In [ ]:
# Find the max value in all the L-buffers
lbuffer_max_value = 0.0
for key in lbuffer_set.keys():
    lbuffer_max_value = max(lbuffer_max_value, np.max(lbuffer_set[key] / gvxr.getUnitOfLength("mm")))


fig, axs = plt.subplots(1, len(lbuffer_set.keys()), figsize=(15, 3))
plt.suptitle("Path length [in mm]")
for i, key in enumerate(lbuffer_set.keys()):
    axs[i].set_title(key)
    im = axs[i].imshow(lbuffer_set[key] / gvxr.getUnitOfLength("mm"),
                       extent=extent,
                       vmin=0.0, vmax=lbuffer_max_value)
fig.colorbar(im, ax=axs.ravel().tolist())
plt.show()

# Simulate the CT scan

Only compute 3 rows of pixels, it's more than enough

In [ ]:
gvxr.setDetectorNumberOfPixels(gvxr.getDetectorNumberOfPixels()[0], 3)

Select the number of projections, make it at least 1000.

In [ ]:
number_of_projections = max(1000, gvxr.getOptimalNumberOfProjectionsCT())

Perform the actual simulations

In [ ]:
rotation_centre_in_mm = gvxr.getNodeAndChildrenBoundingBoxCentre("sample", "mm")
rotation_axis = gvxr.getDetectorUpVector()
number_of_projections = 3 * gvxr.getOptimalNumberOfProjectionsCT()

gvxr.computeCTAcquisition(
    os.path.join(output_path, "projections"), # The path where the X-ray projections will be saved
    "", # The path where the screenshots will be saved
    number_of_projections, # The total number of projections to simulate
    0, # The rotation angle corresponding to the first projection
    False, # A boolean flag to include or exclude the last angle
    360, # The rotation angle corresponding to the last projection
    50, # The number of white images used to perform the flat-field correction
    *rotation_centre_in_mm, # The location of the rotation centre
    unit, # The corresponding unit of length
    *rotation_axis # The rotation axis
)

Read the simulated data with CIL

In [ ]:
reader = gVXRDataReader(gvxr.getProjectionOutputPathCT(),
                        gvxr.getAngleSetCT(),
                        rotation_axis,
                        rotation_centre_in_mm)

data = reader.read()

In [ ]:
print("data.geometry", data.geometry)

Show the geometry in 3D

In [ ]:
show_geometry(data.geometry)

Apply the minus log transformation (use use white_level=1.0 as the flat-field correction is already applied)

In [ ]:
data_corr = TransmissionAbsorptionConverter(white_level=1.0)(data)

In [ ]:
data_corr.reorder(order='astra')

We only want to reconstruct the slice in the middle of the volume

In [ ]:
image_geometry = data_corr.geometry.get_ImageGeometry()

image_geometry.voxel_num_z = 1
print("Image geometry", image_geometry)

In [ ]:
# Perform the reconstruction with CIL
fbp = FBP(image_geometry, data_corr.geometry)
fbp.set_input(data_corr)
reconstruction = fbp.get_output()

Save the reconstructed CT images

In [ ]:
writer = TIFFWriter(data=reconstruction,
                    file_name=os.path.join(output_path, "recons-" + str(number_of_projections), "slice_"),
                    compression="uint16")

writer.write()

Show the CT slice

In [ ]:
plt.figure()
plt.imshow(reconstruction.array,
    origin='upper',
    extent=(0.0, image_geometry.voxel_size_x * reconstruction.shape[1],
        0.0, image_geometry.voxel_size_y * reconstruction.shape[0]),
    cmap="gray"
)
plt.xlabel("Pixel position [" + unit + "]")
plt.ylabel("Pixel position [" + unit + "]")
plt.show()

## Generate the labels

1. Resize the cropped image of labels without interpolation so that it is using the same pixel size as the CT
reconstruction
2. Pad it with 0s so that it is the same number of pixels as the CT reconstruction
3. Save the image

In [ ]:
voxel_size = [image_geometry.voxel_size_x, image_geometry.voxel_size_y]
scaling_factors = [voxel_size[0] / original_pixel_size[0], voxel_size[1] / original_pixel_size[1]]
print(original_pixel_size, voxel_size, scaling_factors)

from skimage.transform import resize

labelled_resized = resize(
    labels_cropped, (round(labels_cropped.shape[0] // scaling_factors[0]), round(labels_cropped.shape[1] //
                     scaling_factors[1])),
    anti_aliasing=False
)

In [ ]:
fig = plt.figure()
im = plt.imshow(labelled_resized,
                origin='upper',
                extent=(0.0, image_geometry.voxel_size_x * labelled_resized.shape[1],
                        0.0, image_geometry.voxel_size_y * labelled_resized.shape[0]),
)
plt.xlabel("Pixel position [" + unit + "]")
plt.ylabel("Pixel position [" + unit + "]")
fig.colorbar(im)
plt.show()

In [ ]:
labels_padded = pad_image(labelled_resized, reconstruction.array.shape)
imwrite(os.path.join(output_path, "labels-resized-padded.tif"), labels_padded)

In [ ]:
fig = plt.figure()
im = plt.imshow(labels_padded,
                origin='upper',
                extent=(0.0, image_geometry.voxel_size_x * labels_padded.shape[1],
                        0.0, image_geometry.voxel_size_y * labels_padded.shape[0]),
)
plt.xlabel("Pixel position [" + unit + "]")
plt.ylabel("Pixel position [" + unit + "]")
fig.colorbar(im)
plt.show()

Plot the labels and the CT data using overlay

In [ ]:
fig = plt.figure()

plt.imshow(reconstruction.array,
           alpha=1.0,
           cmap='gray',
           origin='upper',
           extent=(0.0, image_geometry.voxel_size_x * labels_padded.shape[1],
                   0.0, image_geometry.voxel_size_y * labels_padded.shape[0]),
           )
plt.imshow(labels_padded,
           alpha=0.5,
           origin='upper',
           extent=(0.0, image_geometry.voxel_size_x * labels_padded.shape[1],
                   0.0, image_geometry.voxel_size_y * labels_padded.shape[0]),
)
plt.xlabel("Pixel position [" + unit + "]")
plt.ylabel("Pixel position [" + unit + "]")
plt.show()

# Cleaning up

Once we have finished, it is good practice to clean up the OpenGL contexts and windows with the following command. Note that due to the object-oriented programming nature of the core API of gVXR, this step is automatic anyway.

In [ ]:
gvxr.destroy()